In [ ]:
%pip install dotenv
%pip install datasets
%pip install scikit-learn
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2


In [ ]:
import sys

import torch

sys.path.append(".")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CPU cores:", os.cpu_count())


In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

# Use your token to log in
login(token=os.getenv("hf_token"))


In [ ]:
from huggingface_hub import snapshot_download
import glob
from finetuning.experiments import (
    run_experiment, DatasetConfig, ExperimentConfig,
    ALLSIDES_EXTENDED_MEDIA_SPLIT, ALLSIDES_EXTENDED_RANDOM_SPLIT,
)
from finetuning.models import BERT, BART, ROBERTA, POLITICS, IDEOLOGY_CLASSIFIER


In [ ]:
ideology_dir = snapshot_download(repo_id=IDEOLOGY_CLASSIFIER)
ideology_pt = glob.glob(f"{ideology_dir}/*.pt")[0]
print(f"Ideology classifier .pt: {ideology_pt}")


In [ ]:
MODELS = [BERT, BART, ROBERTA, POLITICS, ideology_pt]

def run_all_models(dataset, output_prefix):
    """Fine-tune every baseline plus our checkpoint on one bias dataset.

    Each run also writes its own <model>_..._test_metrics.json under output_prefix.
    """
    os.makedirs(output_prefix, exist_ok=True)
    results = {}
    for model in MODELS:
        cfg = DatasetConfig(custom_dataset=dataset)
        exp = ExperimentConfig(patience=3, num_epochs=15, save_model=False)
        print(f"\n{'='*60}")
        print(f"Model: {model}  |  Dataset: {dataset}")
        print('='*60)
        results[str(model)] = run_experiment(
            model=model,
            loc=output_prefix,
            dataset_config=cfg,
            experiment_config=exp,
        )
    return results


In [ ]:
media_results_undersampling = run_all_models(
    dataset=ALLSIDES_EXTENDED_MEDIA_SPLIT,
    output_prefix="results_undersampling/media_split",
)


In [ ]:
random_results_undersampling = run_all_models(
    dataset=ALLSIDES_EXTENDED_RANDOM_SPLIT,
    output_prefix="results_undersampling/random_split",
)
